# Pilot: detection error vs. uplink payload (MIDV-Holo + MIDV-DynAttack)

Steps: (1) get code, data and the public 5-fold splits, (2) describe the data and check
identity leakage, (3) run the pilot, (4) read the pre-registered go / no-go decision.

Data: Zenodo record 17079529 (`cropped.tar.gz`, CC BY-SA). Splits: public protocol of
Pouliquen et al. (ICDAR 2024/2025). Both are downloaded at run time and are not redistributed.

In [ ]:
!git clone -q https://github.com/haitham-shiaibth-chasib/rate-constrained-id-pad.git
%cd rate-constrained-id-pad
!pip -q install -U "pillow>=11.3" scikit-image pyyaml
!pip -q install -e .

In [ ]:
# Data: cropped ROIs of MIDV-Holo and MIDV-DynAttack (size is shown by wget)
!mkdir -p data/midvdynattack
!wget -c -O data/cropped.tar.gz "https://zenodo.org/records/17079529/files/cropped.tar.gz?download=1"
!tar -xzf data/cropped.tar.gz -C data/midvdynattack
!find data/midvdynattack -maxdepth 3 -type d -name origins

In [ ]:
# Locate the folder that holds origins/ and fraud/ (archive layout may add a top folder)
import pathlib, yaml
roots = [p.parent for p in pathlib.Path("data/midvdynattack").rglob("origins") if (p.parent / "fraud").is_dir()]
CROPPED = str(roots[0]); print("cropped root:", CROPPED)

In [ ]:
# Public 5-fold splits
!git clone -q --depth 1 https://github.com/EPITAResearchLab/pouliquen.25.icdar.git /content/epita
!rm -rf data/splits_kfold_s0 && cp -r /content/epita/data/splits_kfold_s0 data/
!ls data/splits_kfold_s0

In [ ]:
# Step 2: describe the data, check leakage (must print empty leakage for every fold)
!python scripts/dataset_report.py --cropped_root "$CROPPED" --split_root data/splits_kfold_s0

Before running step 3, look at `source_jpeg_bytes_median` above and adjust `budgets` in
`configs/pilot.yaml` so that the largest budget is near the source size per clip and the
smallest is far below it. Gate thresholds must not be changed after seeing results.

In [ ]:
cfg = yaml.safe_load(open("configs/pilot.yaml"))
cfg["cropped_root"] = CROPPED
yaml.safe_dump(cfg, open("configs/pilot_colab.yaml", "w"), sort_keys=False)
# quick check on one fold first
!python -m idpad.pilot --config configs/pilot_colab.yaml --folds 0

In [ ]:
# all five folds
!rm -f results/pilot/results.csv
!python -m idpad.pilot --config configs/pilot_colab.yaml
!python scripts/plot_pilot.py

In [ ]:
import json, pandas as pd
from IPython.display import Image, display
print(json.dumps({k: v for k, v in json.load(open("results/pilot/gate.json")).items() if k != "per_budget"}, indent=2))
display(Image("results/pilot/eer_vs_bytes.png"))

In [ ]:
# Download results to send back
!zip -qr pilot_results.zip results/
from google.colab import files; files.download("pilot_results.zip")